In [9]:
import pandas as pd
import numpy as np
import pickle
import json

import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [10]:
df = pd.read_csv("data/IMDB Dataset.csv")

df['review'][0]
print(df.columns)
df.head(15)

Index(['review', 'sentiment'], dtype='object')


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
5,"Probably my all-time favorite movie, a story o...",positive
6,I sure would like to see a resurrection of a u...,positive
7,"This show was an amazing, fresh & innovative i...",negative
8,Encouraged by the positive comments about this...,negative
9,If you like original gut wrenching laughter yo...,positive


In [11]:
# Function to clean the reviews, removing unnecessary text/symbols
def clean_review(text):
    text = re.sub(r'<[^>]*>', ' ', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    text = text.lower()
    words = text.split () 

    cleaned_text = " ".join(words)

    return cleaned_text

df['review'] = df['review'].apply(clean_review)

df['review'][0]

'one of the other reviewers has mentioned that after watching just oz episode you ll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not a show for the faint hearted or timid this show pulls no punches with regards to drugs sex or violence its is hardcore in the classic use of the word it is called oz as that is the nickname given to the oswald maximum security state penitentary it focuses mainly on emerald city an experimental section of the prison where all the cells have glass fronts and face inwards so privacy is not high on the agenda em city is home to many aryans muslims gangstas latinos christians italians irish and more so scuffles death stares dodgy dealings and shady agreements are never far away i would say the main appeal of the show is due to the fact that it goes where other shows wouldn t dare forget pretty p

In [12]:
# Change sentiment from text into 1's and 0's
df['sentiment'] = df['sentiment'].astype('category').cat.codes

df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,1
1,a wonderful little production the filming tech...,1
2,i thought this was a wonderful way to spend ti...,1
3,basically there s a family where a little boy ...,0
4,petter mattei s love in the time of money is a...,1


In [13]:
# 5000 features to balance model and memory usage
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=10000)

X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
# Fit and use models to predict
models = {"Logistic Regression": LogisticRegression(max_iter=1000),
          "Random Forest": RandomForestClassifier(n_estimators=100),
          "XGBoost": XGBClassifier(n_estimators=100, max_depth=6)}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    acc = accuracy_score(y_test, predictions)

    results.append({"Model": name, "Accuracy": acc})

    print(f"--{name}--")
    print(f"Accuracy: {acc:.2%}")
    cm = confusion_matrix(y_test, predictions)
    print(f"Confusion Matrix:\n{cm}")

--Logistic Regression--
Accuracy: 90.13%
Confusion Matrix:
[[6613  798]
 [ 682 6907]]
--Random Forest--
Accuracy: 85.50%
Confusion Matrix:
[[6400 1011]
 [1164 6425]]
--XGBoost--
Accuracy: 86.89%
Confusion Matrix:
[[6308 1103]
 [ 864 6725]]


In [16]:
# Save best model and vectorizer

best_model = models["Logistic Regression"]

with open('models/sentiment_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)



In [17]:
# Check the "most" positive and negative words

model = models["Logistic Regression"]
feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]
word_weights = dict(zip(feature_names, coefficients))

with open('models/word_weights.json', 'w') as f:
    json.dump(word_weights, f)

coef_df = pd.DataFrame({'word': feature_names, 'weight': coefficients})

top_positive = coef_df.sort_values(by='weight', ascending=False).head(10)
top_negative = coef_df.sort_values(by='weight', ascending=True).head(10)

print("Top 10 Positive Words:")
print(top_positive)
print("\nTop 10 Negative Words:")
print(top_negative)

Top 10 Positive Words:
           word    weight
3299      great  7.327865
2556  excellent  6.163896
6150    perfect  4.846192
9795  wonderful  4.840522
278     amazing  4.512714
1199  brilliant  4.389904
7915   the best  3.693092
3664  hilarious  3.671768
3101        fun  3.617407
7599     superb  3.613490

Top 10 Negative Words:
           word    weight
9836      worst -7.737685
858         bad -7.595464
841       awful -6.917707
1151     boring -6.495346
8365  the worst -6.078685
9400      waste -5.673490
6298       poor -5.652162
7734   terrible -5.612439
5634    nothing -4.836592
2303       dull -4.802380
